In [ ]:
{
  "nbformat": 4,
  "nbformat_minor": 0,
  "metadata": {
    "colab": {
      "name": "Laboratory_COLAB_DimensionalityReduction.ipynb",
      "provenance": [],
      "collapsed_sections": []
    },
    "kernelspec": {
      "name": "python3",
      "display_name": "Python 3"
    }
  },
  "cells": [
    {
      "cell_type": "markdown",
      "metadata": {
        "id": "JwrQdV29Aucb",
        "colab_type": "text"
      },
      "source": [
        "## Dimensionality Reduction via PCA, t-SNE ##\n",
        "Fabio Scotti fabio.scotti@unimi.it\n",
        "\n",
        "Course: Intelligent systems for industry, supply chain and environment\n",
        "\n"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {
        "id": "3iVlt2KF8Qfa",
        "colab_type": "code",
        "colab": {}
      },
      "source": [
        "# General imports\n",
        "import numpy as np\n",
        "import matplotlib.pyplot as plt\n",
        "\n"
      ],
      "execution_count": 0,
      "outputs": []
    },
    {
      "cell_type": "markdown",
      "metadata": {
        "id": "JfumlS3I-Vqk",
        "colab_type": "text"
      },
      "source": [
        "Loading the digits dataset\n",
        "\n",
        "Each datapoint is a 8x8 image of a digit"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {
        "id": "V9iisxcq8yaT",
        "colab_type": "code",
        "outputId": "df84b401-d190-437a-94f9-25ba0ef315f7",
        "colab": {
          "base_uri": "https://localhost:8080/",
          "height": 413
        }
      },
      "source": [
        "from sklearn.datasets import load_digits\n",
        "digits = load_digits()\n",
        "digits.data.shape\n",
        "print( digits.data.shape ) \n",
        "\n",
        "# let's understand the data we have\n",
        "print( digits.data[0].shape ) \n",
        "print(digits.data[0]) \n",
        "print(digits.target[0])\n",
        "\n",
        "import matplotlib.pyplot as plt \n",
        "plt.gray() \n",
        "plt.matshow(digits.images[0]) \n",
        "plt.show() "
      ],
      "execution_count": 2,
      "outputs": [
        {
          "output_type": "stream",
          "text": [
            "(1797, 64)\n",
            "(64,)\n",
            "[ 0.  0.  5. 13.  9.  1.  0.  0.  0.  0. 13. 15. 10. 15.  5.  0.  0.  3.\n",
            " 15.  2.  0. 11.  8.  0.  0.  4. 12.  0.  0.  8.  8.  0.  0.  5.  8.  0.\n",
            "  0.  9.  8.  0.  0.  4. 11.  0.  1. 12.  7.  0.  0.  2. 14.  5. 10. 12.\n",
            "  0.  0.  0.  0.  6. 13. 10.  0.  0.  0.]\n",
            "0\n"
          ],
          "name": "stdout"
        },
        {
          "output_type": "display_data",
          "data": {
            "text/plain": [
              "<Figure size 432x288 with 0 Axes>"
            ]
          },
          "metadata": {
            "tags": []
          }
        },
        {
          "output_type": "display_data",
          "data": {
            "image/png": "iVBORw0KGgoAAAANSUhEUgAAAPoAAAECCAYAAADXWsr9AAAABHNCSVQICAgIfAhkiAAAAAlwSFlzAAALEgAACxIB0t1+/AAAADh0RVh0U29mdHdhcmUAbWF0cGxvdGxpYiB2ZXJzaW9uMy4yLjEsIGh0dHA6Ly9tYXRwbG90bGliLm9yZy+j8jraAAAL1UlEQVR4nO3df6hX9R3H8ddrptVS0laL0MiMIUSw/IEsitg0w1a4f5YoFCw29I8tkg3K9s/ov/6K9scIxGpBZqQljNhaSkYMtprXbJnaKDFSKgsNsz+U7L0/vsdhznXPvZ3P537v9/18wBe/997vPe/3vdfX95zz/Z5z3o4IARhs3xrrBgCUR9CBBAg6kABBBxIg6EACBB1IoC+CbnuJ7bdtv2N7TeFaj9k+ZHtXyTqn1bvc9jbbu22/ZfuewvXOs/2a7Teaeg+UrNfUnGD7ddvPl67V1Ntv+03bO21vL1xrqu1Ntvfa3mP7uoK1Zjc/06nbUdurO1l4RIzpTdIESe9KmiVpkqQ3JF1dsN6NkuZK2lXp57tM0tzm/hRJ/y7881nS5Ob+REmvSvpB4Z/x15KekvR8pd/pfkkXV6r1hKRfNPcnSZpaqe4ESR9KuqKL5fXDGn2BpHciYl9EnJD0tKSflCoWEa9IOlxq+Wep90FE7GjufyZpj6TpBetFRBxrPpzY3IodFWV7hqRbJa0rVWOs2L5QvRXDo5IUESci4tNK5RdJejci3utiYf0Q9OmS3j/t4wMqGISxZHumpDnqrWVL1plge6ekQ5K2RETJeg9LulfSlwVrnCkkvWh7yPbKgnWulPSxpMebXZN1ti8oWO90yyVt6Gph/RD0FGxPlvSspNURcbRkrYg4GRHXSpohaYHta0rUsX2bpEMRMVRi+V/jhoiYK+kWSb+0fWOhOueot5v3SETMkfS5pKKvIUmS7UmSlkra2NUy+yHoByVdftrHM5rPDQzbE9UL+fqIeK5W3WYzc5ukJYVKXC9pqe396u1yLbT9ZKFa/xURB5t/D0narN7uXwkHJB04bYtok3rBL+0WSTsi4qOuFtgPQf+npO/ZvrJ5Jlsu6U9j3FNnbFu9fbw9EfFQhXqX2J7a3D9f0mJJe0vUioj7I2JGRMxU7+/2UkTcUaLWKbYvsD3l1H1JN0sq8g5KRHwo6X3bs5tPLZK0u0StM6xQh5vtUm/TZExFxBe2fyXpr+q90vhYRLxVqp7tDZJ+KOli2wck/S4iHi1VT7213p2S3mz2myXptxHx50L1LpP0hO0J6j2RPxMRVd72quRSSZt7z586R9JTEfFCwXp3S1rfrIT2SbqrYK1TT16LJa3qdLnNS/kABlg/bLoDKIygAwkQdCABgg4kQNCBBPoq6IUPZxyzWtSj3ljX66ugS6r5y6z6h6Me9cayXr8FHUABRQ6YsT3QR+FMmzZtxN9z/PhxnXvuuaOqN336yE/mO3z4sC666KJR1Tt6dOTn3Bw7dkyTJ08eVb2DB0d+akNEqDk6bsROnjw5qu8bLyLif34xY34I7Hh00003Va334IMPVq23devWqvXWrCl+QthXHDlypGq9fsCmO5AAQQcSIOhAAgQdSICgAwkQdCABgg4kQNCBBFoFvebIJADdGzbozUUG/6DeJWivlrTC9tWlGwPQnTZr9KojkwB0r03Q04xMAgZVZye1NCfK1z5nF0ALbYLeamRSRKyVtFYa/NNUgfGmzab7QI9MAjIYdo1ee2QSgO612kdv5oSVmhUGoDCOjAMSIOhAAgQdSICgAwkQdCABgg4kQNCBBAg6kACTWkah9uSUWbNmVa03mpFT38Thw4er1lu2bFnVehs3bqxa72xYowMJEHQgAYIOJEDQgQQIOpAAQQcSIOhAAgQdSICgAwkQdCCBNiOZHrN9yPauGg0B6F6bNfofJS0p3AeAgoYNekS8IqnuWQcAOsU+OpAAs9eABDoLOrPXgP7FpjuQQJu31zZI+ruk2bYP2P55+bYAdKnNkMUVNRoBUA6b7kACBB1IgKADCRB0IAGCDiRA0IEECDqQAEEHEhiI2Wvz5s2rWq/2LLSrrrqqar19+/ZVrbdly5aq9Wr/f2H2GoAqCDqQAEEHEiDoQAIEHUiAoAMJEHQgAYIOJEDQgQQIOpBAm4tDXm57m+3dtt+yfU+NxgB0p82x7l9I+k1E7LA9RdKQ7S0RsbtwbwA60mb22gcRsaO5/5mkPZKml24MQHdGtI9ue6akOZJeLdEMgDJan6Zqe7KkZyWtjoijZ/k6s9eAPtUq6LYnqhfy9RHx3Nkew+w1oH+1edXdkh6VtCciHirfEoCutdlHv17SnZIW2t7Z3H5cuC8AHWoze+1vklyhFwCFcGQckABBBxIg6EACBB1IgKADCRB0IAGCDiRA0IEEBmL22rRp06rWGxoaqlqv9iy02mr/PjNijQ4kQNCBBAg6kABBBxIg6EACBB1IgKADCRB0IAGCDiRA0IEE2lwF9jzbr9l+o5m99kCNxgB0p82x7sclLYyIY8313f9m+y8R8Y/CvQHoSJurwIakY82HE5sbAxqAcaTVPrrtCbZ3SjokaUtEMHsNGEdaBT0iTkbEtZJmSFpg+5ozH2N7pe3ttrd33SSAb2ZEr7pHxKeStklacpavrY2I+RExv6vmAHSjzavul9ie2tw/X9JiSXtLNwagO21edb9M0hO2J6j3xPBMRDxfti0AXWrzqvu/JM2p0AuAQjgyDkiAoAMJEHQgAYIOJEDQgQQIOpAAQQcSIOhAAsxeG4WtW7dWrTfoav/9jhw5UrVeP2CNDiRA0IEECDqQAEEHEiDoQAIEHUiAoAMJEHQgAYIOJEDQgQRaB70Z4vC6bS4MCYwzI1mj3yNpT6lGAJTTdiTTDEm3SlpXth0AJbRdoz8s6V5JXxbsBUAhbSa13CbpUEQMDfM4Zq8BfarNGv16SUtt75f0tKSFtp8880HMXgP617BBj4j7I2JGRMyUtFzSSxFxR/HOAHSG99GBBEZ0KamIeFnSy0U6AVAMa3QgAYIOJEDQgQQIOpAAQQcSIOhAAgQdSICgAwkMxOy12rO05s2bV7VebbVnodX+fW7cuLFqvX7AGh1IgKADCRB0IAGCDiRA0IEECDqQAEEHEiDoQAIEHUiAoAMJtDoEtrnU82eSTkr6gks6A+PLSI51/1FEfFKsEwDFsOkOJNA26CHpRdtDtleWbAhA99puut8QEQdtf1fSFtt7I+KV0x/QPAHwJAD0oVZr9Ig42Px7SNJmSQvO8hhmrwF9qs001QtsTzl1X9LNknaVbgxAd9psul8qabPtU49/KiJeKNoVgE4NG/SI2Cfp+xV6AVAIb68BCRB0IAGCDiRA0IEECDqQAEEHEiDoQAIEHUjAEdH9Qu3uF/o1Zs2aVbOctm/fXrXeqlWrqta7/fbbq9ar/febP3+wT8eICJ/5OdboQAIEHUiAoAMJEHQgAYIOJEDQgQQIOpAAQQcSIOhAAgQdSKBV0G1Ptb3J9l7be2xfV7oxAN1pO8Dh95JeiIif2p4k6dsFewLQsWGDbvtCSTdK+pkkRcQJSSfKtgWgS2023a+U9LGkx22/bntdM8jhK2yvtL3ddt1TuwAMq03Qz5E0V9IjETFH0ueS1pz5IEYyAf2rTdAPSDoQEa82H29SL/gAxolhgx4RH0p63/bs5lOLJO0u2hWATrV91f1uSeubV9z3SbqrXEsAutYq6BGxUxL73sA4xZFxQAIEHUiAoAMJEHQgAYIOJEDQgQQIOpAAQQcSGIjZa7WtXLmyar377ruvar2hoaGq9ZYtW1a13qBj9hqQFEEHEiDoQAIEHUiAoAMJEHQgAYIOJEDQgQQIOpDAsEG3Pdv2ztNuR22vrtEcgG4Me824iHhb0rWSZHuCpIOSNhfuC0CHRrrpvkjSuxHxXolmAJQx0qAvl7ShRCMAymkd9Oaa7kslbfw/X2f2GtCn2g5wkKRbJO2IiI/O9sWIWCtprTT4p6kC481INt1XiM12YFxqFfRmTPJiSc+VbQdACW1HMn0u6TuFewFQCEfGAQkQdCABgg4kQNCBBAg6kABBBxIg6EACBB1IgKADCZSavfaxpNGcs36xpE86bqcfalGPerXqXRERl5z5ySJBHy3b2yNi/qDVoh71xroem+5AAgQdSKDfgr52QGtRj3pjWq+v9tEBlNFva3QABRB0IAGCDiRA0IEECDqQwH8An6mM7cqa+WgAAAAASUVORK5CYII=\n",
            "text/plain": [
              "<Figure size 288x288 with 1 Axes>"
            ]
          },
          "metadata": {
            "tags": [],
            "needs_background": "light"
          }
        }
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {
        "id": "5eV9480T96Eh",
        "colab_type": "text"
      },
      "source": [
        "Let's plot just two features to see the correlation"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {
        "id": "X6ilkYda9mXi",
        "colab_type": "code",
        "outputId": "9733c452-d1af-41a4-f8e6-f4093f6f93bf",
        "colab": {
          "base_uri": "https://localhost:8080/",
          "height": 283
        }
      },
      "source": [
        "plt.scatter(digits.data[:, 1], digits.data[:, 2],\n",
        "            c=digits.target, edgecolor='none', alpha=0.5,\n",
        "            cmap=plt.cm.get_cmap('jet', 10))\n",
        "plt.xlabel('Feature 1')\n",
        "plt.ylabel('Feature 2')\n",
        "plt.colorbar();"
      ],
      "execution_count": 3,
      "outputs": [
        {
          "output_type": "display_data",
          "data": {
            "image/png": "iVBORw0KGgoAAAANSUhEUgAAAWQAAAEKCAYAAAAl5S8KAAAABHNCSVQICAgIfAhkiAAAAAlwSFlzAAALEgAACxIB0t1+/AAAADh0RVh0U29mdHdhcmUAbWF0cGxvdGxpYiB2ZXJzaW9uMy4yLjEsIGh0dHA6Ly9tYXRwbG90bGliLm9yZy+j8jraAAAgAElEQVR4nO3dd3Rc1bX48e+eoi5ZtiT3IjdswOCCMR1MTDHBgCFAgIRHNyQUh+Qlv4SXkJC2HglJaHkEh5aE3iFOQkswLcHGNu4G494tWbJ6m7J/f8zIljwzkmzNnRmN92etWdYcXZ19RpK37px7zz6iqhhjjEk+V7IHYIwxJsQSsjHGpAhLyMYYkyIsIRtjTIqwhGyMMSnCErIxxqQIS8jGGOMgEZktIitEZKWIfKujYy0hG2OMQ0RkHHADMAUYD8wQkVGxjreEbIwxzjkcmK+qDarqB94DLop1sCdhw+qG4uJiLS0tTfYwjDE9wKJFi3arakl3+pgyapRWNzR0etyaHTtWAk1tmuao6pw2z1cAvxCRIqAR+DKwMFZ/PSIhl5aWsnBhzNdgjDF7icim7vZR3dDAw7NmdXrc6Xfd1aSqk2N9XlVXi8jdwFtAPbAECMQ63qYsjDHGQar6qKoeo6qnAnuANbGO7RFnyMYY01OJSF9VLRORoYTmj4+PdawlZGOMcdZL4TlkH3CzqlbFOtASsjHGOEhVT+nqsTaHbIwxKcKxM2QReQyYAZSp6rg27bcCNxO60vg3Vf1evGJu3AgLFkBLCxx5JEyYACLx6j01PbhpHm+tD73Is0YotwybmpC4H6zbwMsffEh9Q4CCggyuP/MMxvbr63jcZn+A77w1n+0NlYi6OWHQIP77xKMdjwvw7JId/P79emoaXYzuH+DBi4bQPz/L8bh+DfBh8xdsDWwnW7I5NmMsQz29HY8LUMHnbOEj/DRRxBiGcTquBJzHBWihgs9oYDcZ5FPEWDLJdzxusjk5ZfEE8CDw59YGETkduAAYr6rNIhK3/8FffAFPPw2tG6CsXQsVFXDGGfGKkHru/OxtXlo8aO/zh3ZAZdPb3DnmTEfjrtixkzlzP6Q54AW8VDXBr174Bw9efzk5WRmOxr76tXlI9jayckLPF1eVc9e8AD+eOtHRuK+t2sn3Xg4QCIQC766Bc/6wjU+/O9LRuAAvN35MuW4GoFphbvMWZnCm40m5nNWs4CmUIAA1bKWeMsZxuaNxFWUj82ikItyyixq2MJLpeMlxNHayOfanTlXfByr3a/4G8L+q2hw+pixe8T76aF8ybjV/Pvh88YqQet78rE9E299XO3/m9OwHH4ST8T51LZk8/P48R+NuqqqBzJ0R7Ut3b3E0LsD979URCLT/71JWlc1Tn+5wNO6uQO3eZNxKCbDQF/POqbjZwvt7k3GrclbSRI2jcespa5OMQ/w0s4f1jsZNBYmeQz4MOEVE5ovIeyJybKwDRWSWiCwUkYXl5eWddlxbG9nm80FTU2R7umhoyoxsa45si7emppao7TV1dY7G3VJTg7gi76l3uZ3/q1vbGH3ua+3uZkfj1gSj/wI3BRsdjQvQQn1EmxKkhSj/2eLIT/TX5ieN/zOHJTohe4A+hO7D+y7wvEj0WV5VnaOqk1V1cklJ56sgR4+ObBswAPLTeNppRP+KyLZ+kW3xNqZ0aESbC+X0o5ydyz156GCamyJ/oN5AL0fjAkwaFtnmcQe59lhn582HefrgIfKP7BD3QEfjAvRmRERbJvnkMcDRuLn0Q6KkpnyH46aCRCfkrcDLGrIACALF8ej49NNhRJvfnz594MIL49Fz6vr55MGUFFbvfV5SWM3PJg9xPO4Np57C8H6Ki9AckUeCHDEim1NHxyxiFTczR47D15K993lzQx9+dnrMN1pxc/8Fwxk1oGHvRWKvJ8jVJ/sZUujsnGaGuDnNexIeWi8eCv2klBMznf9ej2Q6hez7S5RBLkdwieMX9bxkM4jjcBOaFhOEYsaSz6BOvrLnE91/4jWenYuUAnNb77IQkZuAgap6p4gcBvwTGKqdDGLy5Mna1VoWFRXQ3Bw6O073OyxavV25BIAz+0xIaNwl23awbMsmTh01htLixFz1h9CdFn9euobi3EwuHBt5FuekD9dXsmJnPRcf3Y/iPGcvYLbl1wCb/HsocGVR4s5LWFyAWnbQQj29KcWVwKULAXw0U4WXPLxkd/4FYSKyqKP6El0xZuBA7WIti27HasvJ296eAaYCxSKyFfgx8BjwmIisAFqAqzpLxgeqqCievfUMiU7ErSYMGsCEQYl/G5npcXPDMYcnPC7AySP6cPKIyIupTvOIm5HeuLyZPGDJmipw4yWHbhVt63EcS8iqGuvemK87FdMYY3oyW6lnjDEpwhKyMcakCEvIxhiTItKq2ltF9Vaef+s3tPiDnDLpbCaN+XLCYj8xdxWLF2xn+Ngibr/C2WW8bb29rIz7HlsGwOxrj+bMo52vJ9Hqyge/YO1WHyccnsVvr0rc3Q5r1u/gmb8vpyDHw/VfPYH83K5fge+OoCqrdu+morGZw4sL6ZuT2LsdTPpz9La3eOnKbW//WvQEL31cRZ0vtHDA4/JxwpB1XH/hrx0f3w3Xv8SQz5fsfb514Gge+MvXyMxwOxr36nsWMnDTm+S4Qnt/NQRz2D7sbJ7477jdhRNVfUOAyd9ZQ2NgX2Gd/MxGlj9whKNxAf7wzPs8/G4WAQ19b/PdDfzuGwOYMt7Z+3Kb/X7u+/hzNlaElhKLwMxxvThzROQiGZNcPfm2t7SZspj36Zq9yRjAH/Ty6fZB1DdHLv+Mpz+8vLxdMgYYvP0LfvLAvx2NC5Czfv7eZAyQ42oga/0Cx+Oef3f7ZAxQ25zN7CfWORrX5wvwp3myNxkD1AZyuOdJ5+s6/GvT1r3JGEJ1U/66qprq5vRfzmu6R0RuF5GVIrJCRJ4RkZglAtNmyqKyOfIezTpfPp+seJWpx3zNsbjL52+jX5T23avjVjcppv6eyEI7A6K0xVtZZfR3Vf9Z6WxdhxVrNlEfjFwZt73W+amD9RWR9RX8AVhfVc3Efs6X4DQJVlcLH83rdjciMgi4DThCVRtF5HngMkLVMCOkTULO81RT62tf0yDT3cRRo852NO6Akb0JRjkZzh7ofH2FqkAhfTztC+rtCRQ6HjcnR6mNknsH9XV2imb44H5kyC5atP0KucJM5wvt9MvPYOUOf7s2ERiYl+t4bJN4+QX5TD1zaucHvvNeV7rzANki4gNygO2xDkybKYvxpS48rvZVv8b2XkNRr8GOxv3u1ZPZWdx+HrEyvy/fvfU4R+MCrM2ZRFD3/QiD6mJ97iTH4/7hm0PxSvvvdZa7mSdvdnYet7BXLtMOa78dmUf8XHOO8yvnpg0fSK/9rh1OGZZJv1y7sGdiU9VtwD3AZmAHUK2qb8U6Pm0u6gG88e85fLpmNf6gm9L+mVz55V8kYHRQWdvIz373H+rWl5M5sJDbbzmOkQOdP1MFuOG+xVSt/gIQCg8fxR9nO5+QAT7+oorb5mynoQEKewlPf7uUocWJudvh6dc/5s35FWR74aoLRnHCxDEJiVvT3MyHW3ZQ2eDj8L75TOzXD9ehUjClB4nHRb3Jgwfqwts6v6gn/++uTcDuNk1zVHVOm7H0Bl4CvgpUAS8AL6rqk1H7S6eEbIwxCU7IHcYSkUuA6ap6Xfj5fwHHq+o3ox2fNlMWxhiTgjYDx4tITrj2+zRgdayDLSEbY4xDVHU+8CKwGFhOKOfOiXV82txlYYwxqUhVf0yo/HCn7AzZGGNSRNol5GZfC7X11Z0f6IC6CucXg0TT1OKnqcXf+YEOaGl2djFIzLhBCAY7P86YnsTJHUMeA2YAZa1bOLX53HcI3ZtXoqq7o339gWpoquNvS76Be3AluCC4NpOj+tzFmCFHxaP7Dn348M9Y/bfHaWluwuvNoPT4czjrhw85Hnd3fSPfuu8lqhd/DkCvSWO4d/ZXKE5AsZ1PFn7Ca59uo7LZTf+cAJeeMJqxhx/peNwyP/yiHJY3g1dgWi78dx9wpd2phTkUOflr/AQwff9GERkCnEXo6mPc/G3xt3GVVqIeUBdIv2ZWVP8oniGi2rZyEctffZiWcE0Dn6+FtR+8xtK//tnx2Lc/8ArVC1aF1vD6A1QvWMV3fv+q43E3b9zAn+bvoLI5tDJvZ4Obh+eto7ba+XcmPy6DpU0QVGgOwt9r4Y9VnX+dMT2BYwlZVd8HKqN86nfA94C43gDt7rsjok2KWvh8y/J4homw+C/3ENzvvbMCn73+uKNxAarCZ8ZtVS6KbIu3T1auIajtF0Q0B4TFy5z9Xu/2w2ctke3zGiLbjOmJEvpGT0QuALap6tIuHDtLRBaKyMLy8vJO+9Zo84kqZLidLfzi8nijt7sScAOLO8qPL1pb3MNGX53mdjh2rN7dtljOpImEJWQRyQHuAO7syvGqOkdVJ6vq5JKSzneeDZYNj9KWzfCBow90qAfkuOt+iNvdvqiOyyUc9dWbHY0L0O+4yDnb/sc7P497woQj8Uj7Nzi5niDHTBjvaNw+HpgY5e/rWVbfx6SJRJ4hjwSGA0tFZCMwGFgsIv3j0fklJ99PcN1QaBCkBYJbCjhx8O/i0XWHiocfxpRr7yQntxcul4vM7FyOmnkTY7800/HYD35zJgPPOBYK8qAgj0FnHcsDN13geNx+AwZx02nDGZYfIMutHFYY5NbpR5Od43xm/GlfOD0Xcl1Q5IGv94L/SkzZEGMc52gtCxEpBebuf5dF+HMbgclducvCalkYY7oqlWpZHCjHzpBF5BngP8AYEdkqItc5FcsYY9KBY1eeVPXyTj5f6lRsY4zpiex2emOMSRGWkI0xJkVYtTdjjNlfSy1smJfwsGmVkL91xz180nwYPncWhzWs4skHv5WQuH9buoIHF6zHlRMk2OjinKHF3HbWiQmJfeGfFrN7dR0AxYfn8cpVidnCqb62njWLF9BYVU5e8SDGTj6WjMyMzr8wDu7bU8479S1kC1xTkMs5+el935sSpIqN1LEDD1n0YTSZFCR7WI4K4mcP62ignAzy6cNovETuOO6Y3vlwydTOj/tDlzY57bK0Scg33H4Pf+19EcHs0CzMxvzDmH7rX3jjgSsdjbt5RyVzvviM3kPr97bNa2xk4MIVXDw54m6/uJp67wJYV0/rspQ9C6s5vWoB786e4mjcpsYmFr78CNqwB4DK7auYv3k1J3/1OiTGKr54+ebO7bxd76b1V3dpUzOVwQq+1qvI0bjJtJ1P2MP6vc+r2MAIzkrrpLyZ96lj197nVWxkJGfjwdmVt8mWNnPIH2cfR1Dav5xlJcfz3qKPHY07+/X3yc2vb9eWldXEY4vWx/iKONpQH9GkUdribd2yZXuTcatA1VY2fbHW0bh7An7ebWj/Mw4Cj1U3ORo3mVqoo4oN7doC+KjgsySNyHn1lLVLxgA+Gtr9UeopRGSMiCxp86gRkZhv3dMmIdd7I88W/OJlxacxt6+Ki+YY30KNXuIivqLV7wg4H7apLnp5tYZqZ8uubfW14NfIM/CqQPoWs/DRiEapw+UjfSsqxXptPpw/2Yg3Vf1cVSeo6gTgGKABeCXW8WmTkAfVbohoy/dVcfP11zgad1Re9Myb43N+N+9gZuSPL1pbvBUPHhHZKEL/4VHa4+iorBwK3ZF/cQ7LSP2d0w9WNr1xEzk3n0u/JIwmMXLpixD5RzYNXvM0YJ2qbop1QNok5OuO99Gned+OHZmBRr7c/I7jce//2jlUbe5NMLjvW7lne29eutH5mhInz+jX/ifohqnnO/9LO2zMKHJHHgcS/k/jclN09Jn0KXF+HvcHfbLJkn1vDfq6A/yypNjxuMniwsMgjsPNvj/8+QykD4clcVTO8pLDAI7BFb46Igi9GUEBQ5I8sqiKW6tShh8drbe+DHimo84crWURLwdSy+KW791DU4sy67ozmHLURIdHts89b3zARxsrGVWUza8vOSthcXdVNfGN11fjcsPjM48iPzdx12mrKvZQtXs3xQMGkFeQl7C4NYEAr9RWUeh2c0Ga32HRKoCPBnbjJZssDo3X7KeZRirIIJ9M8rv8dXGpZTFmoC58qAu1LKZ1rZaFiGQA24EjVXVXzOPSLSEbYw5tKZqQLwBuVtUOz9bSZsrCGGNS2OV0Ml0BlpCNMcZRIpILnAm83NmxabMwxBhjUpGq1gNduuJtZ8jGGJMi0uoMuaKmgpeevRIVP5OnzOaY8ecmLPYP/vEai2pgeHYLD59/ScLitvga+XzFPADGjJtKhjc7YbEXVe1kZXUNU4qKGJuXuKXLPoUNLZApMNS77+67RGikEh/15NAXD5mJC5wsqlC9BfyNUFgKnkPgNSeRYwlZRB4DZgBlrVs4icivgfOAFmAdcI2qxmV51yuv3cVO3kcGh+4aWbDlNyxc8BA33jA3Ht136NjnPmRt7SkowoJa+PC5Jcw7azQlvZ3dY27z5hWsf/wuvPW1APw7/3FGXHUnQ4c6W0PDHwzyzfkrmL8hD/DgclVz3hHb+enRRzkaF2CzD56phsbwrcj9PXBlYWiPPScFCbCFD6llOwAu3AzkWAqJ3Fw3bfgaYemTULMt9NyTCeMuhT4jkzuuNObkr/ETwPT92t4Gxqnq0cAa4AfxCraz8T3Eu+8WPnFBsF9jvLqP6ZpXXuCL2iPRNiuLttUO45q333I89hevPrg3GQN4a2tY+9r/OR73ha3rwsk4JBiE11bksqhqp6NxVeG12n3JGGCnH95NwIraPazbm4whlKC38wl+mp0PniybPtiXjAH8zbD6tdAP3DjCsYSsqu8Dlfu1vaWq/vDTjwntPB0f2ZHvW8WjPP/S9+IWIpo1vuhv1bdo/F5aLJnbIldgeqO0xdv8sujFfP61s8LRuLVBqPBHtm/wORoWgAbKItqCBGjE2decVHsiyxHQXAONafyakyyZF/WuBf4R65MiMqt1OWJ5eXnnvUX5j6pBOPJIZ+eRC6mO2l4g0dvjyVcQuWLLX+B8ScZBedF/bUrznJ1fzHFBtFIdvRPwW+wl+krEjBjtaSG7d2Sb2wsZXV81Zw5MUhKyiPwPoRT6VKxjVHWOqk5W1cklJSWd9umqzGb/RYeuSjhy7GndHG3HnjjzLPKz2iffrIxGbhrlfGLMO+0CtM0VLRUh/7QLHY97zfDh9Mlvadc2tLiRCwc6W1zII3BKTmTbqc5O1QPQh9ERF/EKKU3rmsQMOyWUgNsacgJ407smcTIl/C4LEbma0MW+aRrHdds33jCXRx+/HF/2LnAJUu3ixhvejlf3MZX0zuWRSeu4a/l6KrQfvWQP1w4N8rVJpzoe+9iTL+PzooFsX/AmKjDo2LMZc7jzcYszc3j6zEE8tGYjm2vgiCIXN40aicfl/N/3k3OgnxtWtUAGcEw29E3Ab3EGuYxkOpWsxUc9ufSjkFLnAydT/gCYfCNsXxS6y6J4LJSMTfao0pqjtSxEpBSY2+Yui+nAb4HTVLUL8xAhVsvCGNNVqVjLoqscO6URkWeA/wBjRGSriFwHPAjkA2+Hq+f/wan4xhjT0zj2Zk9VL4/S/KhT8YwxJm60FlrmJTxsWq3UM8aYuMjLh5OmduHA+O46bbUsjDEmRVhCNsaYFJE2UxbrN+/igmfXsmPnCDToorC4nOt6r+SOm7/qeOzv3/0kE1Y9w8CGCsqzevHR0HP47S9i7vQdN9urA5x4/zpq6kN7jxXkBvj3bSMZ2MvteOw6drCLZTRTTTZF9Gci2fRxPK4x6SxtzpC/8twqtm4eQ6DFS9DvpnJnf+bsHu943Lkffc6Zix9iSF0Z7mCA/g2VzFzzLL/84xuOxz7x/nVU1WcQxE0QN1X1GZx4/zrH4zZTwybep5FKggSop4yNvEuABKxhNqaHEZFCEXlRRD4TkdUickKsY9MmIe+oiKxAVb27mO/97s+Oxv3g2SfI8rdfteYOBtCFrzoaF6C6IfINTuvZsqNx2YTSvsBMgBZq2ep4bGN6oPuAN1R1LDAeWB3rwLSZsoAEFsVtw0X0yldCcjaP1YQUB47+2jRJr9mYVCUivYBTgasBVLWFUPnhqNLmDLlfn/URbflFlfzq9v9yNO7ki66k2dN+vX9AXAQmznA0LkB+VmRFpYLsKFWW4qwXpch+vzpuvBTEsXifMT1EcWsRtPBj/+V9w4Fy4HER+VREHgnvsRdV2iTkVy4by4Aha3F7A4hLKey3i8t6fex43K+cPo43xl3HjtwiVGB3Vi9eHXExP7rpPMdjz7t5BAXZLQhBhCAF2S28+01nC/wAZFLAUE4mi14IQg5FDGMqbjIcj21MitndWgQt/Jiz3+c9wCTgIVWdCNQD34/VmaO1LOLFalkYY7oqLrUsJg3Uhe91oZZFQce1LESkP/CxqpaGn58CfF9Vo9YFTpszZGOMSTWquhPYIiJjwk3TgFWxjk+ji3rGGJOSbgWeEpEMYD1wTawDLSEbY4yDVHUJ0KUpFJuyMMaYFGEJ2RhjUoRjUxYi8hihrZrK2uwY0gd4DigFNgKXquqeeMW84ZpbmKLryMDPMtcwfvPYI/HqukOLNpXx+3v/Qq/aCmqzCpg56yJmHH1YQmI/umYnb2zbAsD0QUO47rD+CYnbQgNb+IB6yujFUAZxAp4E3fZWxy5q2IwLL70Zkd772plDipNnyE8A0/dr+z7wT1UdDfyTDu7HO1Czr53F9b55TPJtZJxvK1c0f8RvrrkoXt3HVF3dwis/+TmTahczkk1MaFrOkt/9lnc+i7KFepx9f9Fantn8GXsC9ewJ1PPM5s+4Y7HztSx8NLCI37OJ99jNatbxJkv4I8EYqxbjqYLP2ci/qGRtOPYbNLDb8bjGJIJjCVlV3wcq92u+APhT+OM/ATPjFe/04Co8GmjXdoJ/LT/75R/jFSKqbz3wF/q7K9q1Fbprefrhlx2NC/Dpnm0RbYsrna8nsY2PaaT9G5satlHOCkfjBglQxvKINqfjGpMoiZ5D7qeqO8If7wT6xTpQRGa1LkcsL+98P9RewcaItsygj41fxLzlLy5c5RVR27PraxyNCxCIUjvCn4B6Eg1Ef80NdHnf2oMSoDlqRbkW6hyNa0yiJO2inoaWCMbMHqo6p3U5YklJSaf9bXMVRrTVuHO45Zudr7bpjrxxo6O21w8Y5GhcgMwolwCitcVbIdGXZ/ch+vciXjxkk0FeRHsOnf9+GNMTJDoh7xKRAQDhf8vi1fG7rolUu3P2Pm8RD3M945l47OHxChHVfTdcyEr3GLRNtbkNOoT7vn21o3EBvj5iTLsfoCvc5rT+TKSIfRctBWEQU+jFUEfjCsJApuBmXzGnTAroy1GOxjUmURytZSEipcDcNndZ/BqoUNX/FZHvA31U9Xud9dPVWhYvvvA2C954mqygj6r8sdx//w+79wIOwP88+ybbP/2MglFDue+GCxMWd3ejn3s/24xb4NYxQynOTtxanz1soI4dFDKcfAYkLG4AH3XswI2XXPpFVJ4zh7Z41LKYMKm3vvXelzo9rl/By92O1VaHCVlECoASVV23X/vRqrqsw45FngGmAsXALuDHwKvA88BQYBOh2972v/AXwYoLGWO6Kh4J+cjJI/S5hb/o9Lij5Iq4JuSYp1MicilwL1AmIl7galX9JPzpJwiVlItJVS+P8alpBzFOY4xJex2917sDOEZVJxAqhvEXEWl9L56c7TmMMSaNdTTh6G69RU1VF4jI6cBcERlCB3dHGGOMOTgdnSHXisjenUPDyXkqocUdRzo8LmOMOeR0dIb8DfabmlDVWhGZDlzq6Ki64dJvX4cIfOXUc7j0gosTFnfhqkXMXbKAk0aP4cxjO786Gy+BQJBPt4U2sZ046HDcbrvjwJieKmZCVtWlMdp9wFOOjeggXfudm2nMqid/YGg2Ze6yubz0/j947jePOh776jn3Mb9oIoHso3l2c5DDlzzEKzd8w/G463dv48OlD+AO12da9UVvTpkwm+FFibsFzRjTMRHZCNQCAcDf0V0ZaXM6VScN5PTeN7XtzVa8eYEOviI+nnznZf5TfAwBlxuAoMvFyuKj+cnTDzke+8Plf96bjAHcuocPl/2pg68wxiTJ6ao6obNb5NImIWfmRV5nzMxXrrjd2aXT/9y4BZXIm04+rXP+RhRXYFNEm0RpM8b0DF1KyCKS3WaTvpQUaIlsC/phyEBn6wMXZ0T/Fvam2dG4AEFyo7RF1nowxjimuLUIWvgR7QxQgbdEZFGMz+/VaUIWkfOAJcAb4ecTROT1gxm5k3zNbnS/crx15S7u/u5PHY1754VXUVjfvh5vdnMd3zp1qqNxAXoXR148LCo+3fG4xpi9drcWQQs/5kQ55mRVnQScA9wsIqfG6qwrZ8g/AaYAVbB3w77hBz5uZ73wm0ep2+WlbpdQXy7U7HDzwm8edzxufn4BcyaXMqF8Cf2rNnN4+XLuGellwtjxjsc+b8K5DB58FUHPaIKe0QwefBXnTviy43GNMV2nqtvC/5YBrxDKp1F1pRKNT1Wrpf08aUouDHn+N4nZsml/E8aO5/kEJOBoTht7Iow9MSmxjTEdE5FcwBW+ZTgXOAuI+ba9Kwl5pYhcAbhFZDRwG/DvuIzWGGPSWz/glfAJrQd4WlXfiHVwVxLyrcD/AM3A08CbwM+7P05jjElvqroe6PLb5w4Tsoi4gb+p6umEkrIxxhiHdHhRT1UDQFBEeiVoPMYYc8jqypRFHbBcRN4G6lsbVfU2x0ZljDGHoK4k5JfDj7gRkduB6wndrbEcuEZVm7rb7y2XfItRn+/A6w+wdXAR2eeO487Zt3a32049+s4cPnxnOVWuQvKDtYyf1JfvXHqH43GDQeXDDzezZMlOACZO7M9JJw3F5bJy1cb0RJ0mZFWNa3EEERlE6E6NI1S1UUSeBy4jtAvJQfvmpd9m8uINe5+PXreTLS82w+zu9Nq5ZduXM/dfW2jyhgr6NLmzmbfUz7Ahz3PxCc4WxZs3byPvv79vqfQ//7kBvz/I6aen3G3ixpgu6MpKvQ0isn7/RzfjeoBsEfEAOcD2bvbHqDWRXQzaUcUtP7qru1136OnnH6XJnd2uzWbgYV8AABZtSURBVO/y8ME78xyNC7BwYeRrjtZmjOkZujJl0bY6URZwCdDnYAOq6jYRuQfYDDQCb6nqW/sfF17zPQtg6NDOt5d3BSPXqrhUaWrxH+xQuyToj15RLuh3fu2MzxcZ2+8PRjnSGHMgGvGzgrKEx+30DFlVK9o8tqnqvcC5BxtQRHoT2nVkODAQyBWRr0eJO6d1fXhJSUmn/W4ZVhzRVl6UzyN3/+xgh9olZ0w/D0+wfdIXlCOPc35TlaOO6hfRNm5cX8fjGpP+8lGmdfqIt65MWUxq85gsIjfRtTPrWM4ANqhqebjY/ctAt9f+Tp19CeuHl+B3CSqwq7iAlacM6263nTpr3FkcN9pHvq8GgBx/PRNKyrnpy7c4Hnv69FEcfXQ/3G7B7RaOProfZ589yvG4xhhndCWx/qbNx35gA93bwmkzcLyI5BCaspgGLOxGfwCcd8ZUzjtjKo89/SzrN+/g59+/vbtddtkPrw6dhc9f+zFHDhpPXnZ2J18RHxkZbi666HDOO+8wALxed0LiGmOc0ZWEfF14+d9eInLQl/FVdb6IvAgsJpTgPwWilaw7KNdecVm8ujpgx406PilxLREbkx66Un7zxS62dZmq/lhVx6rqOFW9UlWdr+ZujDEpLuYZsoiMBY4EeonIRW0+VUDobgtjjDFx1NGUxRhgBlAInNemvRa4wclBGWPMoShmQlbV14DXROQEVf1PAsdkjDGHpK5c1PtURG4mNH2xd6pCVa91bFQHafYVX8XV7COgQnaWn7uffi0hcesbG7jrtadZX9lI/3wPd57/Vfr2Oui1M6YTjVRQwxZceClkOF5ykj0kYzoULmW8ENimqjNiHdeVi3p/AfoDZwPvAYMJTVuklO9eMZPKsgDllULlHti208N3L52ZkNgz//g4b+/OYV2wiI+qe3HxEy9QVl2ZkNiHmj2sYz1vU85qdrGMtfydRvYke1jGdGY2sLqzg7qSkEep6o+A+nChoXOB47o5uLirrs1AA22WKyvs2uPll3f8ytG4v/rr0+x29W7XVuvO5xevP+9o3ENRkAC7WIq22dIxgI9ylidxVMZ0TEQGE8qbnW762ZWE7Av/WyUi44BeQMqtz21uiawdEfQH2bVxkaNxv9gd/exsW12Lo3EPRQGa8RN5h2QzNUkYjTEAFIvIwjaPWVGOuRf4HtBpoZmuzCHPCdef+BHwOpAH3HkgI06EnCyo3S8Hur0uvvw1Z+shTxkyiIWfNUS0jy7KdTTuochDFl5y8NH++51NUZJGZAy7VXVyrE+KyAygTFUXicjUzjrrSnGhR1R1j6q+p6ojVLWvqv7hwMbsvKy8AC5vm5fjgv59Wjj73JMdjXvDGTMplYp2bSXBPfzwvMsdjXsoElwMZDIu9q1M9JJDX45K4qiM6dBJwPkishF4FviSiDwZ6+CuFBfqJyKPisg/ws+PEJHr4jXaePndUy/Ta9gQBvX1MaAkQOGADO5+NjF3Wbxy8618c2wOU3L3cOUIN2/ddjO52Xbl3wn5DGI05zGIKQzhJEYzgwzykj0sY6JS1R+o6mBVLSW0Ece/VDWiumWrrkxZPAE8zr5dp9cAzwGPdm+o8ffAH3+btNg3nDHTVsskiJdsejMy2cMwJu66clGvWFWfJzwhrap+IHpVdmOMMVGp6ryO7kGGriXkehEpIrQhKSJyPFAdh/EZY4xpoytTFt8mdHfFSBH5CCgBLnZ0VMYYcwjqqNrbUFXdrKqLReQ0QsWGBPg8vNOHMcaYOOroDPlVYFL44+dU9SsJGE+33TTzJiTo4aHXH0xoXJ/Cbj/0dkNWVyaC4qh1YUQmBYkNbEyaqg3CvPrEx+0oIUubj0fEM6iIFBJaRjiO0Nz0td2tKDf7qtvYs8FDQ3XoJX3tuNvo1T/A/732+26PtzNLm+AfddAUBK/AablwcgLuevPRyBY+pIHdAORQwhBOwktitpAyJl15g9kMqB+X8LgdnctpjI/j4T7gDVUdC4ynC0U3OlO7PYuG6n1L9ZqbgtTs6s5erF1TFYDXakPJGEJnyu/UwZYETOrsZNHeZAzQQDk7Wex8YGOMIzpKyONFpEZEaoGjwx/XiEitiBx08QAR6QWcSvg+ZlVtUdWqg+2vVXNNU0RbU2OAGy+8sbtdd2htCwSj/Ln6PAGbUtWwLaKtNkqbMaZn6KhAvVM7Zw4HyoHHRWQ8sAiYrartZmzCRTpmAQwdOrTTTl1uF/vX7nC5IKsgMz6jjiEnxp+03ATMI3vIiqjr4MbZ12uMcU6CLz8BoT8Ck4CHVHUiUA98f/+DVHWOqk5W1cklJSWddppd5G0/6w3kFuVw35/uj8ugYxmTAcX7/VnLc8H4BOw6WMzYKG2HOx/YGOOIZCTkrcBWVZ0ffv4i++7mOGhz/vprioZmkdMrg+w8D4WDcrni9nO7222n3ALXFMIJOTDEC8dkw3W9Y585x1MRYxjMCeTRnzz6M4QTKeIw5wMbYxzh/FWv/ajqThHZIiJjVPVzYBqwKh59P/z6PfHo5oDluuDsJNW3KaSUQkqTE9wYE1cJT8hhtwJPiUgGsB64JknjMMaYlJGUhKyqS4CYRZ2NMeZQlIw5ZGOMMVFYQjbGmBRhCdkYYxwiIlkiskBElorIShG5q6Pjk3VRzxF3Tz2DYRs34AkE2FnSj5pJp3HHI79yPG5ZoI6/1m9js99HP7eHGbkDGOrp5XjcpCrbBf9+DyrKYeBgOGkqFKT5azbmwDUDX1LVOhHxAh+KyD9U9eNoB6dNQv7VadMYv3r53uejtm5gW2MD4GxCblY/91ZtpCoQWj9d7vex1reZH/YeSR93mu6rV1cLzzwOzeHl6hXlsGkDXH8LeNLmV8qYblNVBerCT73hR8zaQGkzZVG6YUNE24DKMm4+N+Z+gnGxuLlsbzJu1RSE/zSXORo3qVYs3ZeMW9VUwdrPkzMeY5KnWEQWtnnM2v8AEXGLyBKgDHi7zaK4CGlzOuMJRm7z51JFvX5H4zZq9O0FG6OMJ23sn4w7azcmfe1W1Q5v4VXVADAhXHb4FREZp6oroh2bNmfIO4v7RrRV5RTwf68+62jcCRlFuPeroSECEzN7Oxo3qcYcEdnm9sDIMYkfizE9RLiq5bvA9FjHpE1C3nz4Sezo3ReVUHaszs7j44ndLpHRqT7uHK4qKKEgnJWzXXBRXi9Gevs4Hjtp+g+Es2ZAVrgQfn4BnH8x5CVp/bgxKUpESsJnxohINnAm8Fms49NmyuJ/n7kXgBvPv4wMj/DAy8/w1QTFnpzZn4kZfakINlDoyiZDnKpcmkImTIZxE6C+LpSQXWnzt92YeBoA/ElE3IROgJ9X1bmxDk6bhNzq4dednaKIxS0u+roPsTNEjwd6FSZ7FMakLFVdBkzs6vF2WmOMMSki7c6QjTGmu2qbYF4S7uK0hGyMMfvJ98PUys6Pey/OcW3KwhhjUkTSzpDDVx0XAttUdUY8+rzmtKuo25MPLUE8Q1w88/aD8ejWGGMSIplnyLOB1fHq7LIzboMFWyj8bDGF65eQ+/4SLjl2dry6N8YYxyUlIYvIYOBc4JG49bmsAk9g39Jd0SB5K1Zx3/+7M14hjDHGUck6Q74X+B4QjHWAiMxqLdhRXl7eaYcZ9RURbZ5APQsWpXGRH2NMWkl4QhaRGUCZqi7q6DhVnaOqk1V1cklJSaf9tuQWRbT53blMOSayxoUxxqSiZJwhnwScLyIbgWeBL4nIk93tVMf3xu/O3vdcXNSNO4LZd/+0u10bY0xCJDwhq+oPVHWwqpYClwH/UtVuFy1+9u0HYcpgqg4/hqoRE2g45Whe+OS+bo/XGGMSJa0Whjz+3p+SPQRjjDloSU3IqjoPmJfMMRhjTKqwlXrGGJMiLCEbY0yKsIRsjDEOEZEhIvKuiKwSkZUi0uHy4bRLyHfechH3/OjCxAcOBqFqD/id3VTVGNOj+IHvqOoRwPHAzSISZVPKkLS5y+KuWV9n1Lk+JpwaQBSee+Vitq/wc/uPXnU++JrV8M7foa42tM/cqdNCWxwZYw5pqroD2BH+uFZEVgODgFXRjk+bM+SR5/rIaWpCNPQ8o8XHwHEJ+HtTVwt/fTH0L0BTI7w1F3Zudz62MSbZiltLPIQfs2IdKCKlhLZzmh/rmLRJyBmByKmCjBYfP7t9prOB162BQCCyfU3cCtkZY1LX7tYSD+HHnGgHiUge8BLwLVWtidVZ2iRkFYnSBo01MesXxUdGZvT2zCxn4xpjegQR8RJKxk+p6ssdHZs2CbnRH/lSGjMz+eWjrzsbeNQYKNhv5+WsbDjyaGfjGmNSnogI8CiwWlV/29nxaZOQr77keapdmTRnevFleKjLymLHmznOB/Z64Ypr4KiJUFQCY46Ey6+BvHznYxtjUt1JwJWEiqgtCT++HOvgtLnLAkJJuZ0LEhS4oBeck6hgxhin1dbCvHnd70dVPwQi51NjSKuEbIwx8ZDvhqm9Oj/Odp02xpg0ZQnZGGNShCVkY4xJEZaQjTEmRST8op6IDAH+DPQDFJijqnHZa+m5mybSP+8L0CB12h8deQMzbv5BPLo2xhjHJeMM+YCqH3XVi7ccRZEuw1fTgK+2icy6jWRvvLfbgzXGmERJxianO1R1cfjjWqC1+lG3lGSuA9X2jbUV/HTm2d3t2hhjEiKpc8gdVT8SkVmtFZTKy8s77Uuj1CHWYJA8d2P3B2qMMQmQtITcWfUjVZ3TWkGppKSk0/78GX0i2rx5GXz7pffjMVxjjHFcUhLygVQ/6qp/fzEeb0Hu3ufeHC+flR8Tj66NMSYhknGXxQFVP+qqO199E4D7LzuN3Aw/1835iNPi1bkxxiRAMmpZtFY/Wi4iS8Jtd6jq3+PR+W3Pxnt1uTHGJEbCE/KBVj8yxphDha3UM8aYFGEJ2RhjHCQij4lImYis6OxYS8jGGOOsJ4DpXTkwrQrU/37WOYzKrsYtyuYGL9fOsXuQjTHJparvhxfBdSptzpCfuvEUTmETw2rLGVyzm+MDu/j7bScle1jGGNNlaZOQR2Y0kOH37X3u0iCDm/bw/nOPJnFUxpg0V9xa4iH8mNWdztImIWe3NEW0Zfh9rH/n8SSMxhhziNjdWuIh/JjTnc7SZg652ZtJpq+lXZvf7UHz+iVpRMaYnqq2toV58zYmPG7aJOTNwVwOd9XjDgYBUIQdOYVc87uXkjwyY0xPk5/fzNSpGzs97r0uLAwWkWeAqYSmN7YCP1bVqHOpaZOQL/6/D3jihpMpzffj0iDlgUy+ct8HyR6WMeYQp6qXd/XYtEnIAFf/8cNkD8EYYw5a2lzUM8aYns4SsjHGpAhLyMYYkyIsIRtjTIpIu4R83p1XcvHd1yV7GAnl8wXw+QLJHoYxppuScpeFiEwH7gPcwCOq+r/d7fP875xG8f9eStZPzgTg6/45ZLy+iccu+kV3u05ZLS0B/vrXz1m5MrQr97hxfZkx4zAyMtxJHpkx5mAk/AxZRNzA74FzgCOAy0XkiO722/eXF1HjKUBFUBGa3Nm0nF/a3W5T2htvrGX58jKCQSUYVJYt28Wbb65N9rCMMQcpGVMWU4C1qrpeVVuAZ4ELuttpvSc/oq3JncX1f/l+d7tOWcuX74rSVpaEkRhj4iEZCXkQsKXN863htnZEZFZrBaXy8vJOOxXVqG0tdY3dGGpq83ojpya83rS7LGDMISNl//eq6pzWCkolJSWdHp/fXB3RlhNo4M/fuM+J4aWEyZMHdqnNGNMzJCMhbwOGtHk+ONzWLQ/nfpuixt14gz7cwQC9fNX4f720u92mtKlTS5k2bThFRdkUFWUzbdpwpk4tTfawjDEHKRl3WXwCjBaR4YQS8WXAFfHo+OGc2/c9cQN3xKPX1OVyCaecMoxTThmW7KEYY+Ig4QlZVf0icgvwJqG0+Ziqrkz0OIwxJtUk5T5kVf078PdkxDbGmFSVshf1jDHmUGMJ2RhjHCQi00XkcxFZKyIdLoywhGyMMQ450JXJlpCNMcY5B7QyWTTKCrdUIyLlwKYD+JJiYLdDw0lV9poPDfaaOzdMVTtfTdYBEXkjHLczWUBTm+dzVHVOm34uBqar6vXh51cCx6nqLdE66xF76h3oN1dEFqrqZKfGk4rsNR8a7DUnhqpOT2S8VjZlYYwxzjmglcmWkI0xxjl7VyaLSAahlcmvxzq4R0xZHIQ5nR+Sduw1HxrsNfcgB7oyuUdc1DPGmEOBTVkYY0yKsIRsjDEpIu0S8oEsU0wHIjJERN4VkVUislJEZid7TIkgIm4R+VRE5iZ7LIkgIoUi8qKIfCYiq0XkhGSPyWkicnv4d3qFiDwjIlnJHpPT0iohO7WBaorzA99R1SOA44GbD4HXDDAbWJ3sQSTQfcAbqjoWGE+av3YRGQTcBkxW1XGELohdltxROS+tEjIObaCaylR1h6ouDn9cS+g/asQehelERAYD5wKPJHssiSAivYBTgUcBVLVFVauSO6qE8ADZIuIBcoDtSR6P49ItIXdpA9V0JSKlwERgfnJH4rh7ge8BwWQPJEGGA+XA4+FpmkdEJDfZg3KSqm4D7gE2AzuAalV9K7mjcl66JeRDlojkAS8B31LVmmSPxykiMgMoU9VFyR5LAnmAScBDqjoRqAfS+vqIiPQm9O52ODAQyBWRryd3VM5Lt4TsyAaqqU5EvISS8VOq+nKyx+Owk4DzRWQjoSmpL4nIk8kdkuO2AltVtfWdz4uEEnQ6OwPYoKrlquoDXgZOTPKYHJduCfmAlimmAxERQnOLq1X1t8kej9NU9QeqOlhVSwn9fP+lqml95qSqO4EtIjIm3DQNWJXEISXCZuB4EckJ/45PI80vZEKaLZ0+RDdQPQm4ElguIkvCbXeE9y006eNW4KnwicZ64Jokj8dRqjpfRF4EFhO6k+hTevAS6q6ypdPGGJMi0m3KwhhjeixLyMYYkyIsIRtjTIqwhGyMMSnCErIxxqQIS8jGUSISEJElbR6lB9HHTCcLJonIGyJSdahUjjOpK63uQzYpqVFVJ3Szj5nAXA5gMYSIeFTV38XDf02oeM2NBzE2Y+LGzpBNwonIMSLynogsEpE3RWRAuP0GEflERJaKyEvhVVonAucDvw6fYY8UkXkiMjn8NcXhZdSIyNUi8rqI/Av4p4jkishjIrIgXJQnauU/Vf0nUJuQF29MBywhG6dlt5mueCVcd+MB4GJVPQZ4DPhF+NiXVfVYVW2t93udqv6b0PL376rqBFVd10m8SeG+TwP+h9DS6inA6YSSelpXSTM9m01ZGKe1m7IQkXHAOODtUIkC3ITKKwKME5GfA4VAHqEl8AfqbVWtDH98FqFCRP8dfp4FDOUQqIlgeiZLyCbRBFipqtG2IHoCmKmqS0XkamBqjD787Ht3t/+2PvX7xfqKqn5+0KM1JoFsysIk2udASeuecCLiFZEjw5/LB3aEpzW+1uZrasOfa7UROCb88cUdxHoTuDVcLQwRmdj94RvjHEvIJqHCW2tdDNwtIkuBJeyrc/sjQrudfAR81ubLngW+G74wN5LQThLfEJFPgeIOwv0M8ALLRGRl+HkEEfkAeAGYJiJbReTsg36BxnSDVXszxpgUYWfIxhiTIiwhG2NMirCEbIwxKcISsjHGpAhLyMYYkyIsIRtjTIqwhGyMMSni/wMRqOsYT4VODAAAAABJRU5ErkJggg==\n",
            "text/plain": [
              "<Figure size 432x288 with 2 Axes>"
            ]
          },
          "metadata": {
            "tags": [],
            "needs_background": "light"
          }
        }
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {
        "id": "bLmdM0oCODud",
        "colab_type": "text"
      },
      "source": [
        "______\n",
        "## Feature extraction with PCA ##"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {
        "id": "wQGxmyQP8UJh",
        "colab_type": "code",
        "outputId": "8211833a-b690-4132-f869-d06d4888df45",
        "colab": {
          "base_uri": "https://localhost:8080/",
          "height": 51
        }
      },
      "source": [
        "from sklearn.decomposition import PCA\n",
        "\n",
        "pca = PCA(2)  # project from 64 to 2 dimensions\n",
        "projected = pca.fit_transform(digits.data)\n",
        "print(digits.data.shape)\n",
        "print(projected.shape)"
      ],
      "execution_count": 4,
      "outputs": [
        {
          "output_type": "stream",
          "text": [
            "(1797, 64)\n",
            "(1797, 2)\n"
          ],
          "name": "stdout"
        }
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {
        "id": "MxyP72u2-Kma",
        "colab_type": "text"
      },
      "source": [
        "Plotting the first 2 principal component (PCA si unsupervised --> no labels are used!) "
      ]
    },
    {
      "cell_type": "code",
      "metadata": {
        "id": "WuTOllYy8qMM",
        "colab_type": "code",
        "outputId": "76bfb1cf-fd9d-4b5e-cdd4-8e2da7309298",
        "colab": {
          "base_uri": "https://localhost:8080/",
          "height": 283
        }
      },
      "source": [
        "plt.scatter(projected[:, 0], projected[:, 1],\n",
        "            c=digits.target, edgecolor='none', alpha=0.5,\n",
        "            cmap=plt.cm.get_cmap('jet', 10))\n",
        "plt.xlabel('component 1')\n",
        "plt.ylabel('component 2')\n",
        "plt.colorbar();"
      ],
      "execution_count": 5,
      "outputs": [
        {
          "output_type": "display_data",
          "data": {
            "image/png": "...\n",
            "text/plain": [
              "<Figure size 432x288 with 2 Axes>"
            ]
          },
          "metadata": {
            "tags": [],
            "needs_background": "light"
          }
        }
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {
        "id": "kS0p8YsA-w2f",
        "colab_type": "text"
      },
      "source": [
        "______\n",
        "## Feature extraction with t-SNE  ##\n"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {
        "id": "smezMv2G-0w-",
        "colab_type": "code",
        "outputId": "eef65106-69cf-4d46-903b-641e311347ca",
        "colab": {
          "base_uri": "https://localhost:8080/",
          "height": 155
        }
      },
      "source": [
        "from sklearn.manifold import TSNE\n",
        "\n",
        "tsne = TSNE(n_components=2, verbose=1, perplexity=40, n_iter=300)\n",
        "tsne_results = tsne.fit_transform(digits.data)  # no y labels\n"
      ],
      "execution_count": 6,
      "outputs": [
        {
          "output_type": "stream",
          "text": [
            "[t-SNE] Computing 121 nearest neighbors...\n",
            "[t-SNE] Indexed 1797 samples in 0.010s...\n",
            "[t-SNE] Computed neighbors for 1797 samples in 0.462s...\n",
            "[t-SNE] Computed conditional probabilities for sample 1000 / 1797\n",
            "[t-SNE] Computed conditional probabilities for sample 1797 / 1797\n",
            "[t-SNE] Mean sigma: 8.394135\n",
            "[t-SNE] KL divergence after 250 iterations with early exaggeration: 61.485268\n",
            "[t-SNE] KL divergence after 300 iterations: 0.960238\n"
          ],
          "name": "stdout"
        }
      ]
    },
    {
      "cell_type": "code",
      "metadata": {
        "id": "_JrMyd8e_bRZ",
        "colab_type": "code",
        "outputId": "0d17d6a3-8ed2-4701-a441-6542b8d89313",
        "colab": {
          "base_uri": "https://localhost:8080/",
          "height": 283
        }
      },
      "source": [
        "plt.scatter(tsne_results[:,0], tsne_results[:,1] , \n",
        "            c=digits.target, edgecolor='none', alpha=0.5,\n",
        "            cmap=plt.cm.get_cmap('jet', 10))\n",
        "plt.xlabel('tsne-2d-one')\n",
        "plt.ylabel('tsne-2d-one')\n",
        "plt.colorbar();\n"
      ],
      "execution_count": 7,
      "outputs": [
        {
          "output_type": "display_data",
          "data": {
            "image/png": "...\n",
            "text/plain": [
              "<Figure size 432x288 with 2 Axes>"
            ]
          },
          "metadata": {
            "tags": [],
            "needs_background": "light"
          }
        }
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {
        "id": "LUqQS4F-DSoX",
        "colab_type": "text"
      },
      "source": [
        "__________\n",
        "Recusive Feature Elimination   \n",
        "wrapper: SVM   (you can try with kNN, LDA, NN, etc.)"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {
        "id": "NH4HsZGFDaXB",
        "colab_type": "code",
        "outputId": "e8686718-ee03-4201-b7cc-3f82a1fff6c6",
        "colab": {
          "base_uri": "https://localhost:8080/",
          "height": 498
        }
      },
      "source": [
        "from sklearn.svm import SVC\n",
        "from sklearn.feature_selection import RFE\n",
        "\n",
        "# Load the digits dataset #digits = load_digits()\n",
        "X = digits.images.reshape((len(digits.images), -1))\n",
        "y = digits.target\n",
        "\n",
        "print(X[0])\n",
        "print(y[0])\n",
        "\n",
        "\n",
        "# Create the RFE object and rank each pixel\n",
        "svc = SVC(kernel=\"linear\", C=1)\n",
        "rfe = RFE(estimator=svc, n_features_to_select=1, step=1)\n",
        "rfe.fit(X, y)\n",
        "ranking = rfe.ranking_.reshape(digits.images[0].shape)\n",
        "\n",
        "# Plotting the results\n",
        "print(\"ranking of the pixel w.r.t the classification capability\")\n",
        "print(ranking)\n",
        "\n",
        "# Plot pixel ranking\n",
        "plt.matshow(ranking, cmap=plt.cm.Blues)\n",
        "plt.colorbar()\n",
        "plt.title(\"Ranking of pixels with RFE\")\n",
        "plt.show()"
      ],
      "execution_count": 8,
      "outputs": [
        {
          "output_type": "stream",
          "text": [
            "[ 0.  0.  5. 13.  9.  1.  0.  0.  0.  0. 13. 15. 10. 15.  5.  0.  0.  3.\n",
            " 15.  2.  0. 11.  8.  0.  0.  4. 12.  0.  0.  8.  8.  0.  0.  5.  8.  0.\n",
            "  0.  9.  8.  0.  0.  4. 11.  0.  1. 12.  7.  0.  0.  2. 14.  5. 10. 12.\n",
            "  0.  0.  0.  0.  6. 13. 10.  0.  0.  0.]\n",
            "0\n",
            "ranking of the pixel w.r.t the classification capability\n",
            "[[64 50 31 23 10 17 34 51]\n",
            " [57 37 30 43 14 32 44 52]\n",
            " [54 41 19 15 28  8 39 53]\n",
            " [55 45  9 18 20 38  1 59]\n",
            " [63 42 25 35 29 16  2 62]\n",
            " [61 40  5 11 13  6  4 58]\n",
            " [56 47 26 36 24  3 22 48]\n",
            " [60 49  7 27 33 21 12 46]]\n"
          ],
          "name": "stdout"
        },
        {
          "output_type": "display_data",
          "data": {
            "image/png": "iVBORw0KGgoAAAANSUhEUgAAAPgAAADwCAYAAAAtgqlmAAAABHNCSVQICAgIfAhkiAAAAAlwSFlzAAALEgAACxIB0t1+/AAAADh0RVh0U29mdHdhcmUAbWF0cGxvdGxpYiB2ZXJzaW9uMy4yLjEsIGh0dHA6Ly9tYXRwbG90bGliLm9yZy+j8jraAAAac0lEQVR4nO3df7RcZX3v8ffnJCQhJBBCgOYHEAQaSrUEjBQWiApFwVKSdqkLFtJoUXrvsi5Y2MsP+0O9t1ZtvRZu8XpXFDRWRH4IBZEiKT+CVAETCAgELgSJJCSEmGCCEEKST//Yz9DJyTln9iTPPmfPnO8ra6/MzN7znWfmzHeeZz/72c+WbUII3alnqAsQQqhOJHgIXSwSPIQuFgkeQheLBA+hi0WCh9DFujbBJd0j6WP9rPs3SXOHoEzHS3pa0iuS5rT53HdKeipDGfr9XDLEHrCMkqZLsqSRVbx+2NGQJrik5yS9lr7wqyV9S9K4ql/X9mm251f9On34n8AVtsfZ/td2nmj7x7ZnVFSuLHqXMf19/2Bn46Xvw+b0/VgnaYGkw5vWf0TS1rS+sVzRx3MbyyO79g47Tx1q8D+yPQ6YCRwFXDrE5anSQcDjQ12IDvMP6fsxFVgJXNlr/U/TD2Zj+Yvez21ajhy0UtdEHRIcANurgR9RJDoAki6RtEzSRklPSPrjpnUfkXSfpC9LWi/pF5JO6yu2pMmSHpX0P9L9N5upreJIOljSvakM/y7pq5K+09/7kPRxSc+kGucWSVPS48uAtwA/SLXJ6D6e+5ykS9N7XS/pm5LGpHXvlrQi3T4kxT863Z8i6SVJ7073j5X0E0kvS3qk8Xgfr3eopIWSfi1praRr+9luvqRPpdtTUzP7E73K0tOrjP8CHNj0fi9qCnm2pF+m1/yr/j7LZrZfA66j6ftRJyP2PMg9Y/crtUi6fbDKVZt9IUnTgNOAu5oeXga8E1gNfBD4jqRDba9K638fmA9MAs4DrpQ01U3jbyUdTPHD8WXb8/p5+YHifBf4D+APgGOA24Bb+nkPJwFfAN5LUVN/GfgecKLtQyQ9B3zM9r8P8FGcDbwP+A3wA+Cv0/Im28skXZw+j1nAN4H5tu+RNBX4IXAOcDtwMvB9SYfbfqnXa/0v4A7gPcAoYFY/ZVoI/DHwv4F3Ac8CJwJfTfd/bHubpOYyniPpnc3vV9L0tPoEYAbw28CDkm60vXSAzwRJewBnAc8MtN1Q8ZZNjD78zFLbbnr4nydVXJw31aEG/1dJG4HngTXAZxorbF9v+wXb22xfCzxNkWQNy21/3fZWigSdDOzftP4I4G7gMwMkd79xJB0IvAP4W9ubbd9HP8mdnA1cZfsh269T7G4c1/TFLuMK28/bXgd8nuJLvQPbX6f4sj+QytuoCT8M3Gb7tvS5LQAWAe/vI8wbFLsNU2xvSu+vLwuBEyT1UCT2PwDHp3XvSuvb8Tnbr9l+BHgEGKjp/JeSXgY2UvwwnNNr/bGppdJYju393Kalun4XAVK5ZRDVIcHn2B4PvBs4nKIWBUDSn0pa0vgDAW9tXk9RswNg+9V0s7mT7myK/bYbWpShvzhTgHVNj0HxQ9SfKcDyplivAL+i2H8sqzn+8hSzP1+n+Ez+Of2gQJGwH2z+YlMkxuQ+nn8RxVfzQUmPS/qzvl7E9jKKFsVMihbVrcALkmawcwm+uun2q2z/N+vty7YnANOB1yhq/mb3257QtNzf+7lNS7VHTtRTbhlEdUhwAGwvBL5F0axF0kEUX+C/APZJf+THKL6QZX0WWAt8V9KInSjWKmCipLFNjx0wwPYvUCQY8Gazch+KH5mymuMfmGLuQMXRhssoOp0+K2liWvU88C+9vth72P5i7xi2V9v+uO0pwJ8D/1fSof2UayHwAWCU7ZXp/lxgb2BJP8/Jdqqi7V8C5wOXS9o9V9x8BD0jyi2DaMgSXNKpFDXbfEmXpIcvA06RdCSwB8UX5KW0/UcpaqtWviBpDUXT+g2Kffc9gG+nJmZptpdTNG8/K+ktkhZTNJlPl3R+H0+5BviopJmpE+3vgQdsP9ffa0gaI+lBFYdwpgCfkzQtJexfAX12fAGXA4tsf4xin/v/pce/A/yRpPdJGiFprIpj7zvs90v6YOr7AFhP8Xlv6+f1FlL82N6b7v8pcHG6/UA/z3mRomOxNEkTJN0g6UlgDk2tn7S78QJFP0lbJM1IrcHGskHSBe3GafEi0UQHSLXpVym+AOcCZ0k6InUCfZtin/cJik6dn6bt3kbR2dXKjcCpjTu2NwN/QrFvflW7SU7RzD8OeBTYSlFj3gJ8QtIRzRumzqS/Ab5PUfsfArTqeXkdOCkdwnkh3f8xRUfWMuDvej9B0uz0Hv97euhC4GhJZ9t+HpgNfJrix3ENMLZ3jOQdwAOSXknv6Xzbz/az7UJgPP+V4JsofhD+3nZ/nXNfAP467Sr8ZT/b9HY5cLvtw1OZ1vZa/4/ARerjKEQfLko9+K8Ai4FptmcCb6fYNbipZJlaE7VsomsoJnyQdBzwWdvvS/cvBbD9hUzxpwO32i5T47cb+1rgSYr90StSrZIr9nJgM/Bh2/3Viu3Em0bRafh54ELbp+9qzKbYzwGzbPdOwF2JuRdFc/8trvCLKem9FB2vx7fcuKSecZM9+m3ldvE33f+lxQP8KGY1VE30qWzfmbSC9jqiBo2kd6RjvT1pt2I2RZP0KPpvmrb7GiMkLaHY/16cI7mTyyg60vprdu8KA3dIWiyp7SZzPw6maHV8U9LDkr6R+jFyO5NidyqvGtbgtelkq7HfAu4BXgH+D3ABxfHjC2xvyPECtrempuMK4HBJu9zykHQ6sMb24l0uYN9OsH00xdiFT0g6MUPMkcDRwNdsH0XRc3/JwE9pj6RRwBnA9TnjpuCxD56sZPve4mm019M8aGz/wPYBtscCv0sx4ONq2zdW8FoH0qsPYRccD5yRmtLfA07SACPw2pV60rG9hmJf9piBn1HKCmBFUwvmBoqEz+k04CHbL+YNq6w1eHNno6Slko6TNFHFePyn0/97t4ozVAn+M+AwFcNAR1E0mQYaQDLkVAzTuhJYavsrGePuK2lCur07cArFPv4usX2p7Wm2p1N8vnfZ/vCuxoXi8J+k8Y3bFCP3HtvVuGm48vPp+DoUo/Ce2NW4vZxFJc1zch8ma+5sPBJYStGaudP2YcCdlGjdDEmC295CccjlRxQFv852lpMwJF1D0fM+Q9IKSefmiEtRI55DURM2DrX0NTqsXZOBuyU9SvHDt8D2rRniVml/4L50aO9B4Ie2c42v/iRwdfo8ZlIcaswi/RidQtFKyixfDZ46G08knViTRlG+TNH/0xiNN5/iMOLAsYaiFz2EbtMzfqpHz/pvpbbddM/fDtiLLmkmMI+i9XIkxSG+84GVacBXo0W5vnG/33KVK34IYUDtHQefJGlR09L7KETLzsZ0GLFl7Vybs8lC6Hjle8jXtjgO3ldn4yXAi5Im214laTLFIKYBRQ0eQhb59sEH6Gy8hWL8P+n/m1vFiho8hFzyHuNudDaOohi2/FGKCvm61HG8HPhQqyCR4CHkIGU9U8z2EvqegOPkduIMeRM94zDHjo5bZexOi1tl7CrLHENV+1bVB95pcauM3Wlxq4xdYYLXb6hqNNFDyEKDXjuXUclAF43c3Ro1vtS23vIaGllugo6R4/YsXYZtr22gZ/fy20/ca0yp7V779Xp236vlEOA3jd2t/B994/p1jN97YusNgT12K//bvG7dWiZOLD/P3xvbyn0nXl63lgltxF376ubS227asJ4xe5b/nH+9YVOp7bZt2kDPmHLfi60bX2Lbpg2lqtyevQ706BPKnfK+6bbzB+100UpqcI0az+gZLTv42jbxuLb6F9py9vuPaL3RTjhqShVnO8KsqeV+CHbGmg2vt95oJ1z10IpK4gL88M5dvujLDn5108WtN2poDHSpmWiih5BFPZvokeAh5DLIEyqWEQkeQi6D3ENeRiR4CDkomughdLca1uClfnIknSrpKRUX1cs6R1YI3UJSqWUwtUzwpjnMT6O41tdZvecDD2G4Ky5N1oEJTjGZ3jO2n00XEfgexdQxIYQGCfWUWwZTmQTvmDnMQxhKdazBs3WypbN0ioH8uw10scgQutNgJ28ZZRK81Bzm6frb8wB6xu4XMzmGYaeOCV6mid5xc5iHMOjUxjKIWtbgtrdIasxhPgK4Ktcc5iF0CzH4+9dllNoHt30bcFvFZQmho3VsgocQWuvpiaGqIXSnIdi/LiMSPIRMookeQpfq6E62EEJrwyfBR+0O0343e9jjZh2YPWbDqJHV/HEuv2NZJXG/dlb5CSXbdf0TqyuJu+T/r60kLsDWLVvzB213uFb98jtq8BCy0HCqwUMYhnIeJpP0HLAR2ApssT1L0kTgWmA68BzwIdvrByxTthKFMIw1Otkyn032Htszm+ZQvwS40/ZhwJ30umZ4XyLBQ8il+rHos4H56fZ8YE6rJ0SCh5CDsp8PbuAOSYubLpi4v+1V6fZqYP9WQWIfPIRM2kjeSZIWNd2fl063bnaC7ZWS9gMWSHqyeaVtS2rZzx8JHkImbST42lbXJrO9Mv2/RtJNFFOnvShpsu1VkiYDa1q9UDTRQ8gl0z64pD0kjW/cBt4LPEYxD8PctNlc4OZWsVrW4JKuAk4H1th+a+vihTD8SMp5mGx/4KbUIhgJfNf27ZJ+Blwn6VxgOdDyCp9lmujfAq4Avr3TxQ1hGMg10MX2s8CRfTz+K6CtS+yWmdHlXknT2wkawnDU1SPZtptVdffqrl0dQm3VL7/zJfh2s6pOOChmVQ3DTlfX4CEMa3GySQjdq7g22VCXYkdlLj54DfBTYIakFamLPoSwHdHTU24ZTGV60c8ajIKE0OmiiR5Ct1I9m+iR4CFkIBj05ncZkeAhZBI1eAhdLPbBQ+hS0jBqomtED6PGj8sed+2GTdljNrzruGqmZJ4zo+WkGzvljmdfqiQuwIUnHFxJ3Kde2FBJXIBfPJ1/2mS7nQGZceGDELpaDfM7EjyEXKIGD6FbxXHwELpXMRa9fhkeCR5CJjXM70jwEHIZNofJQhh2ano+eJnTRQ+QdLekJyQ9Lun8wShYCJ2kcT54mWUwlanBtwCfsv1Qmqt5saQFtp+ouGwhdJAOHeiSroW0Kt3eKGkpMBWIBA+hSQ3zu7198DR98lHAA1UUJoRO1pE1eIOkccD3gQts7zCouHnaZI3dJ1sBQ+gInTzQRdJuFMl9te0b+9qmedrkEfscHNMmh2GlmPChfpf6K3NtMgFXAkttf6X6IoXQmepYg5f5yTkeOAc4SdKStLy/4nKF0HEklVraiDdC0sOSbk33D5b0gKRnJF0raVSrGC0T3PZ9tmX792zPTMttpUsZwnBQ8hh4m7X8+cDSpvtfAv7J9qHAeqDlFOb122kIoQOJcrV32Rpc0jTgD4FvpPsCTgJuSJvMB+a0ihNDVUPIJPM++GXARcD4dH8f4GXbW9L9FRTjUQYUNXgImfRIpRZgkqRFTct5zXEknQ6ssb14V8sUNXgIGbQ56eJa27MGWH88cEbqzB4D7AlcDkyQNDLV4tOAla1eKGrwEDLpUbmlFduX2p5mezpwJnCX7bOBu4EPpM3mAje3ilVNDW7YuiX/LJfr11c3q+ohk/LPAgvwy3WvVhL3bfuOb73RTvrkjT+vJO5NH//9SuIC7H3jwuwxt72xua3tB2Go6sXA9yT9HfAwxfiUAUUTPYRMqshv2/cA96TbzwLHtPP8SPAQMhDFobK6iQQPIZMaztgUCR5CFm0OQx0skeAhZCBgRA2r8EjwEDKpYQUeCR5CLh3ZRJc0BrgXGJ22v8H2Z6ouWAidZChmTC2jTA3+OnCS7VfSzC73Sfo32/dXXLYQOkpPDTO8zKyqBl5Jd3dLS0zJFEIv9Uvv8nOyjQAWA4cCX7Uds6qG0KSuveilTjaxvdX2TIozWI6R9Nbe20g6r3H6m1/fmLucIdRbyckeBrsjrq2zyWy/THFGy6l9rJtne5btWRpd3YkQIdRVHS9dVObaZPtKmpBu7w6cAjxZdcFC6DR1rMHL7INPBuan/fAe4Drbt1ZbrBA6i+jQsei2H6W4XFEIYQAdOdAlhFBO/dI7EjyELKR6HiaLBA8hk2iih9DFapjfkeAh5CDUmWPRd4Zff5Wtv3g0e9zxxxySPWbDXc+uqSTuvc+8XEncw/bbo5K4AJ8+6bBK4m7Zuq2SuAC8si5/zG1tzAzcwWeThRBKiH3wELpYHa8iEgkeQgZ1PZssEjyETGqY35HgIeRQnClWvwyv425DCB0p18UHJY2R9KCkRyQ9Lulz6fGDJT0g6RlJ10oa1bJMu/62QgiQ9XzwxjyIRwIzgVMlHQt8Cfgn24cC64FzWwWKBA8hg+J0UZVaWnGhr3kQTwJuSI/PB+a0ilU6wSWNkPSwpDgXPIQ+9JRcykj5tgRYAywAlgEv296SNlkBTG0Vp51OtvOBpcCebTwnhGFBUjuHySZJWtR0f57tec0b2N4KzEyzKd0EHL4z5So7q+o04A+BzwMX7swLhdDt2uhEX2t7VpkNbb8s6W7gOGCCpJGpFp8GrGz1/LIthsuAi4B+BxNvN6vqltdKhg2he2TsRe9rHsSlFBOefiBtNhe4uWWZSrzY6cAa24sH2m67WVVH7t4qbAhdJWcnG8U8iHdLehT4GbAgzYN4MXChpGeAfYArWwUq00Q/HjhD0vuBMcCekr5j+8NlShrCcJFrnEt/8yDafhY4pp1YLWtw25fanmZ7OnAmcFckdwi9lGyeD/Zw1hiqGkImquG0i20luO17gHsqKUkIHUzAyBoOG4saPIRM6niySSR4CBl07JVNQgglxJxsIXS3YTOragjDzfBqoveMgDHjsocdM6a636PZR0ypJO4JB06qJO7mLdVNQbzfnqMriVtlmZnwW/ljjmjn+yZGRA0eQncSsQ8eQvcaglFqZUSCh5BJdLKF0KWiiR5Cl4saPIQuVsP8jgQPIQeJzj1MJuk5YCOwFdhSdj6pEIaT+qV3ezX4e2yvrawkIXSwxpRNdRNN9BAyqV96l59V1cAdkhZLOq/KAoXQqTJeuiibsjX4CbZXStoPWCDpSdv3Nm+QEr9I/tF75S1lCLWnWk74UKoGt70y/b+G4ioLO8zsuN20ybuNzVvKEGpO5L10US5l5kXfQ9L4xm3gvcBjVRcshE6TcV70bMo00fcHbkrNj5HAd23fXmmpQug06tA52dJk60cOQllC6FiNJnrdxGGyEDLpyBo8hFBO/dK7nq2KEDpSruPgkg6QdLekJyQ9Lun89PhESQskPZ3+37tVrEjwEDIo9sFVailhC/Ap20cAxwKfkHQEcAlwp+3DgDvT/QFFgoeQRblDZGUOk9leZfuhdHsjxbXBpwKzgflps/nAnFaxqtkHHzkK9j0oe9jnn/919pgN1/98ZSVxf/zM+krinnZENbO1AsyeVM0MswueerGSuABTjvjt7DFXPDymre3b6GObJGlR0/15tuf1HVPTKS4l/ACwv+1VadVqikPYA4pOthAyaDTRS1pb5pRrSeOA7wMX2N7Q3Etv25LcKkY00UPIoWQHW9laXtJuFMl9te0b08MvSpqc1k8G1rSKEwkeQiYZe9EFXAkstf2VplW3AHPT7bnAza1iRRM9hEyU70j48cA5wM8lLUmPfRr4InCdpHOB5cCHWgWKBA8hg5zXJrN9H/2Pmzm5nViR4CFkElM2hdDFMjbRsynVySZpgqQbJD0paamk46ouWAidpNFEL7MMprI1+OXA7bY/IGkUEFO2hLAd1bIGb5ngkvYCTgQ+AmB7M7C52mKF0GGGYELFMso00Q8GXgK+KelhSd9IUzeFEJqo5DKYyiT4SOBo4Gu2jwJ+Qx9nsUg6T9IiSYv8+sbMxQyh3kRx6aIyy2Aqk+ArgBW2H0j3b6BI+O1sN6vq6PE5yxhCZ6hhFd4ywW2vBp6XNCM9dDLwRKWlCqEDqeS/wVS2F/2TwNWpB/1Z4KPVFSmEzlTHTrZSCW57CRBXFA1hADXM7xjJFkI2NczwSPAQMij6z+qX4ZHgIeQwBMNQy4gEDyGXSPAQulWHjkUPIZTTsYfJ2rblDVi/qvV2bdr02n7ZYzbM/p3JlcTdvGVbJXF/8tyGSuIC7DVqt0ri/s6+e1YSF2DMmPxf5XYmcBiKceZlRA0eQi41zPBI8BAyiX3wELpYHCYLoVvVdCc8EjyETKKJHkKXEsPpMFkIw1AN87v1hA+SZkha0rRskHTBYBQuhI7SoTO6PGV7pu2ZwNuBV4GbKi9ZCB0m54wukq6StEbSY02PTZS0QNLT6f+9W8Vp9+qiJwPLbC9v83khdL3MFz74FnBqr8cuAe60fRhwJ31MfrpDmdooP8CZwDV9rdhuVtU3ftNm2BC6QMYmuu17gXW9Hp4NzE+35wNzWsUpneBpPrYzgOv7KdB/zaq6W0ybHoaXxoQPFU+6uL/txkkeq4H9Wz2hnV7004CHbL+4MyULoau1d2WTSZIWNd2fZ3teOy9n25Lcart2Evws+mmehxDa6iBfa3tnJjF9UdJk26skTQbWtHpC2auL7gGcAty4E4UKYXio/jDZLcDcdHsucHOrJ5SdNvk3wD47X64Qul3eGV0kXQO8m6I5vwL4DPBF4DpJ5wLLgQ+1ihMj2ULIoHF98Fxsn9XPqpPbiRMJHkIuNRyrGgkeQiZxNlkIXSzOJguhi9Uwv5Hd8lh5+0Gllyh6+cqYBKzNXojOi1tl7E6LW2XsduIeZHvfMhv+3lFv9213/aRU0AMmjlm8k8fB21ZJDV72QwGQtKiKN9tpcauM3Wlxq4xdZZnrWIdHEz2EDHIfJsslEjyETKKTrW9tDbLv4rhVxu60uFXGrqzMdTxMVkknWwjDzZFHvd0/Wnh/qW0n7zWqszvZQhiO6ld/R4KHkIXaOx980ESCh5CJapjhkeAhZFK/9I4EDyGbGlbgkeAh5JF3wodcIsFDyKCu1yZrd170EEIHiRo8hEzqWINHgoeQg6CnhhkeCR5CBkNw4dBSIsFDyKWGGR4JHkImcZgshC5Ww13wSPAQcqlhfkeCh5BNDTM8EjyEDIo52eqX4TGjSwgZSLqdYkrmMtbaPrXK8jREgofQxWIseghdLBI8hC4WCR5CF4sED6GLRYKH0MX+E9e44HGp3OTKAAAAAElFTkSuQmCC\n",
            "text/plain": [
              "<Figure size 288x288 with 2 Axes>"
            ]
          },
          "metadata": {
            "tags": [],
            "needs_background": "light"
          }
        }
      ]
    }
  ]
}